# 08 — Index F (Digital-Frontier)

**Objetivo.** Predecir Index_F — la serie más difícil — explotando network_metrics.

**Pistas del enunciado.** Digital-Frontier — volatilidad extrema tipo cripto. Atado a `network_metrics` (on-chain). **Domina el RMSE junto a A** — el mayor impacto potencial en la clasificación.

**Nivel de esfuerzo.** ALTO. Explorar arquitecturas. Features de network. El clip defensivo (`clip_logret=0.5`) es crítico aquí para evitar divergencia en rollouts largos.

**Estrategia.** LSTM + features de network alineadas → backtest → CNN-LSTM → backtest → ensemble 5 seeds del ganador. Monitorizar si el clip se activa (RMSE enorme = modelo diverge).

**Qué esperamos.** RMSE alto en cualquier caso (cripto es impredecible). Si las network_metrics tienen señal, podemos mejorar sobre el baseline. Comparar siempre con baseline_drift.

**Qué NO hace.** No usa macro_factors (es para C). No usa news.

**Inputs.** `data/train_indices.csv`, `data/train_network_metrics.csv`, `data/test_network_metrics.csv`, `results/baselines.json`

**Output esperado.** `models/{OWNER}_Index_F.keras`, `results/index_F.json`

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

OWNER = "oscar"   # <-- cada miembro pone su nombre aquí
IDX   = "Index_F"

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import (
    load_hackathon_data, make_window_dataset, make_temporal_split,
    align_aux_features, build_model, train_model, train_ensemble,
    backtest_autoregressive, plot_history, plot_rollout,
    DATA_DIR, VAL_DAYS, V_IN_SHARED, RANDOM_SEED
)

os.makedirs('models',  exist_ok=True)
os.makedirs('results', exist_ok=True)
MODEL_PATH = f'models/{OWNER}_{IDX}.keras'

## 1. Carga + diagnóstico + referencia baselines

In [ ]:
data  = load_hackathon_data(DATA_DIR)
idx   = data['train_indices']
serie = idx[IDX].dropna().values

plt.figure(figsize=(13, 3))
plt.plot(serie, lw=0.8)
plt.title(f'{IDX} — precios brutos (alta volatilidad esperada)')
plt.tight_layout()
plt.show()

with open('results/baselines.json') as f:
    baselines = json.load(f)
print(f'Baselines para {IDX}:')
for bl, rmse in baselines[IDX].items():
    print(f'  {bl:<15} RMSE = {rmse:,.0f}')

best_bl_name = min(baselines[IDX], key=baselines[IDX].get)
best_bl_rmse = baselines[IDX][best_bl_name]

## 2. Alineamiento de network_metrics

Traer de `00_carga_y_EDA` las columnas con mayor correlación con F.

In [ ]:
if 'train_network' not in data:
    raise RuntimeError('train_network_metrics.csv no encontrado — necesario para F')

# Ajustar NET_FEATURES con las columnas identificadas en 00_carga_y_EDA
NET_FEATURES = list(data['train_network'].columns)   # <-- filtrar al resultado del EDA

aux_train = align_aux_features(idx, data['train_network'], NET_FEATURES)
print('Features network alineadas:', list(aux_train.columns))

aux_values = aux_train.loc[idx.index].values   # (T, k) — alineado con serie completa

## 3. Dataset con features de network

In [ ]:
serie_train, _ = make_temporal_split(serie, val_days=VAL_DAYS, v_in=V_IN_SHARED)
aux_train_only = aux_values[:len(serie_train)]

X, y = make_window_dataset(serie_train, V_IN_SHARED, use_log_rets=True,
                            aux_features=aux_train_only)
cut  = int(len(X) * 0.8)
X_tr, y_tr = X[:cut], y[:cut]
X_v,  y_v  = X[cut:], y[cut:]

n_features = X.shape[2]
print(f'X_tr={X_tr.shape}  n_features={n_features}  (1 precio + {n_features-1} network)')

## 4. Candidato 1 — LSTM + network

In [ ]:
import tensorflow as tf
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

model_lstm = build_model('lstm', V_IN_SHARED, n_features=n_features, dropout=0.2)
hist_lstm  = train_model(model_lstm, X_tr, y_tr, X_v, y_v, epochs=300)
plot_history(hist_lstm, title=f'{IDX} — LSTM + network')

In [ ]:
predict_fn_lstm = lambda x: model_lstm.predict(x, verbose=0).ravel()[0]

# clip_logret=0.5 activo por defecto — especialmente importante para F (cripto)
bt_lstm = backtest_autoregressive(
    predict_fn_lstm, serie, log_ret_mode=True, aux_data=aux_values
)
print(f'LSTM+network  RMSE backtest = {bt_lstm["rmse"]:,.0f}  |  dir_acc = {bt_lstm["dir_accuracy"]:.2%}')
plot_rollout(serie, bt_lstm['preds'], index_name=f'{IDX} — LSTM+network', val_days=VAL_DAYS)

## 5. Candidato 2 — CNN-LSTM + network

In [ ]:
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

model_cnn = build_model('cnn_lstm', V_IN_SHARED, n_features=n_features)
hist_cnn  = train_model(model_cnn, X_tr, y_tr, X_v, y_v, epochs=300)

predict_fn_cnn = lambda x: model_cnn.predict(x, verbose=0).ravel()[0]
bt_cnn = backtest_autoregressive(
    predict_fn_cnn, serie, log_ret_mode=True, aux_data=aux_values
)
print(f'CNN-LSTM+network  RMSE backtest = {bt_cnn["rmse"]:,.0f}')

## 6. Ensemble de semillas del ganador (si hay tiempo)

In [ ]:
best_tipo = 'lstm'   # cambiar a 'cnn_lstm' si ese fue el ganador

ens = train_ensemble(
    best_tipo, X_tr, y_tr, X_v, y_v,
    v_in=V_IN_SHARED, n_features=n_features,
    n_seeds=5, epochs=300
)
bt_ens = backtest_autoregressive(
    ens['predict_fn'], serie, log_ret_mode=True, aux_data=aux_values
)
print(f'Ensemble  RMSE backtest = {bt_ens["rmse"]:,.0f}')
plot_rollout(serie, bt_ens['preds'], index_name=f'{IDX} — Ensemble {best_tipo}', val_days=VAL_DAYS)

## 7. Decisión final y guardado

In [ ]:
resultados = {
    'lstm_network':     bt_lstm['rmse'],
    'cnn_lstm_network': bt_cnn['rmse'],
    'ensemble':         bt_ens['rmse'],
}
for k, v in resultados.items():
    print(f'  {k:<20}  RMSE = {v:,.0f}')

ganador    = min(resultados, key=resultados.get)
best_rmse  = resultados[ganador]
print(f'Ganador: {ganador}  vs  baseline {best_bl_name}: {best_bl_rmse:,.0f}')

_tipo_map = {'flat': 'baseline_flat', 'drift': 'baseline_drift', 'random_walk': 'baseline_rw'}

if best_rmse < best_bl_rmse:
    if ganador == 'ensemble':
        ens['models'][0].save(MODEL_PATH)
        approach = 'nn_ensemble'
    elif ganador == 'lstm_network':
        model_lstm.save(MODEL_PATH)
        approach = 'nn'
    else:
        model_cnn.save(MODEL_PATH)
        approach = 'nn'
    final_rmse = best_rmse
    final_path = MODEL_PATH
    notes      = f'{ganador} + {n_features-1} features network, clip_logret=0.5'
else:
    approach   = _tipo_map[best_bl_name]
    final_rmse = best_bl_rmse
    final_path = None
    notes      = f'Ningún modelo supera baseline {best_bl_name}'

info = {
    'index':              IDX,
    'owner':              OWNER,
    'approach_type':      approach,
    'strategy':           ganador,
    'rmse_backtest_252d': final_rmse,
    'model_path':         final_path,
    'log_ret_mode':       True if approach in ('nn', 'nn_ensemble') else False,
    'v_in':               V_IN_SHARED if approach in ('nn', 'nn_ensemble') else None,
    'ghost_source_index': None,
    'ghost_lag':          None,
    'notes':              notes
}

with open('results/index_F.json', 'w') as f:
    json.dump(info, f, indent=2)

print('Guardado: results/index_F.json')
print(json.dumps(info, indent=2))